# NB01 – Daten Laden

**Grid-Arbitrage** · Batteriespeicher-Arbitrage im Schweizer Strommarkt

**Gruppe:** SC26_Gruppe_2 | **Verantwortlich:** Patrik Neunteufel | **Datum:** März 2026

---

*Lädt [ENTSO-E](../organisation/O_02_Glossar.ipynb#g-entsoe) Spot-Preise und CH-Netzlast von externen APIs in `data/raw/`.*


|  [← NB00 Business Case](00_Business_Case.ipynb) | [↑ Übersicht ↑](../organisation/O_01_Project_Overview.ipynb) | [NB02 Daten Bereinigung →](02_Daten_Bereinigung.ipynb) |
|:---|:---:|---:|


## Inhaltsverzeichnis<a id='toc_NB_01'></a>

[Einleitung](#einleitung_NB_01)  
[Initialisierung](#initialisierung_NB_01)  
[Datensatz 1: ENTSO-E Day-Ahead Spot-Preise CH](#datensatz-1-entso-e-day-ahead-spot-preise-ch_NB_01)  
[Datensatz 2: CH Netzlast (Systemlast)](#datensatz-2-ch-netzlast-systemlast_NB_01)  
[Fazit](#fazit_NB_01)  
[Abschluss](#abschluss_NB_01)  

---
## Einleitung <a id='einleitung_NB_01'></a>

[↑ Inhaltsverzeichnis](#toc_NB_01)

Laden der Pflicht-Rohdaten via ENTSO-E Transparency Platform:

- **Datensatz 1** — Day-Ahead-Spot-Preise CH (stündlich, EUR/MWh)
- **Datensatz 2** — Systemlast CH (stündlich, MW)

Download erfolgt jahresweise mit Retry-Logik, um API-Rate-Limits und temporäre
503-Fehler zu überbrücken. Output in `data/raw/` als CSV. Der tatsächlich
geladene Datenzeitraum wird nach `../sync/transfer.json` geschrieben, damit
NB02–K_99 damit arbeiten können.


## Initialisierung<a id='initialisierung_NB_01'></a>

[↑ Inhaltsverzeichnis](#toc_NB_01)

**Abhängigkeiten prüfen:** Fehlende Pakete (`pandas`, `numpy`, `requests`, `entsoe-py`)
werden automatisch per `pip` installiert, falls nicht vorhanden.


In [1]:
# ── lib/ aus Projekt-Root erreichbar machen + lib-Imports ───────────────────
# Notebook liegt in einem Unterordner (kuer/, experimental/, notebooks/,
# organisation/). Damit 'from lib.xxx import ...' funktioniert, muss der
# Projekt-Root vorne in sys.path stehen. autoreload sorgt dafür, dass
# Änderungen in lib/*.py ohne Kernel-Restart übernommen werden.
import sys, os
_PROJECT_ROOT = os.path.abspath('..')
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

# lib-Imports (einmal zentral — in allen folgenden Zellen verfügbar)
from lib.plotting import show_source
from lib.data_fetchers  import fetch_entsoe_yearly
from lib.io_ops   import load_transfer, save_transfer, log_dataindex, needs_download, log_missing, final_check

print(f'lib-Pfad aktiv: {_PROJECT_ROOT}/lib')


lib-Pfad aktiv: C:\Users\patri\Documents\docker\python\OWN\Grid-Arbitrage/lib


In [2]:
# ── Check, if used Libraries need to be installed ───────────────────────────
import subprocess, sys
for imp, pkg in [('pandas','pandas'),('requests','requests'),('numpy','numpy'),
                 ('entsoe','entsoe-py')]:
    try: __import__(imp)
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])


**Imports**
  
Import needed libraries


In [3]:
import os, sys, warnings, json
import numpy  as np
import pandas as pd
import requests
from datetime import datetime
warnings.filterwarnings('ignore')

# Versionen anzeigen für Reproduzierbarkeit
# print(f"Os           Version: {os.__version__}")
# print(f"Sys          Version: {sys.__version__}")
# print(f"Warnings     Version: {warnings.__version__}")
print(f"Numpy        Version: {np.__version__}")
print(f"Pandas       Version: {pd.__version__}")
print(f"Requests     Version: {requests.__version__}")
print(f"📅 Zuletzt ausgeführt am: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")

Numpy        Version: 2.2.6
Pandas       Version: 2.3.3
Requests     Version: 2.32.5
📅 Zuletzt ausgeführt am: 26.04.2026 um 12:51:19


**Setup – Konfiguration & Verzeichnisstruktur:** 

Lädt `../sync/config.json` (SSOT), setzt alle
Pfade, legt fehlende Verzeichnisse an und definiert `needs_download()` als Reload-Guard.


In [4]:
# ── Projektstruktur & Konfiguration ──────────────────────────────────────────────────
# ── ../sync/config.json laden (Single Source of Truth) ───────────────────────────────
# Schalter NIE direkt hier setzen — immer in ../sync/config.json anpassen.
_cfg_path = '../sync/config.json'
if not os.path.exists(_cfg_path):
    raise FileNotFoundError(
        '../sync/config.json nicht gefunden. '
        'Bitte ins Projektverzeichnis legen (siehe NB00b Sektion 5).')
with open(_cfg_path) as _f:
    CFG = json.load(_f)

# ── Aliases (nur lesend — Quelle bleibt ../sync/config.json) ─────────────────────────
MODE         = CFG['mode']            # 'data' | 'sim'
FORCE_RELOAD = CFG['force_reload']    # dict mit allen Keys

# ── Datenzeitraum ────────────────────────────────────────────────────────────
_start   = CFG['daten']['start_year']
_end     = CFG['daten']['end_year']
START_YEAR = int(_start)
END_YEAR   = datetime.now().year if str(_end) == 'heute' else int(_end)
YEARS      = list(range(START_YEAR, END_YEAR + 1))
print(f'../sync/config.json geladen | MODE={MODE}')
print(f'Datenzeitraum: {START_YEAR}–{END_YEAR} | Jahre: {YEARS}')

# ── Verzeichnisstruktur ───────────────────────────────────────────────────────
DATA_FOLDER      = '../data'
OUTPUT_FOLDER    = '../output'
DIR_RAW          = os.path.join(DATA_FOLDER, 'raw')
DIR_PROCESSED    = os.path.join(DATA_FOLDER, 'processed')
DIR_INTERMEDIATE = os.path.join(DATA_FOLDER, 'intermediate')
DATAINDEX        = '../sync/dataindex.csv'
SZ_AKTIV         = CFG['szenarien']['gleichzeitigkeit_aktiv']  # SSOT

# Nur benötigte Ordner anlegen:
# - sim/ wird nur von NB01b angelegt (SIM-Mode)
# - output/kuer/ existiert nicht mehr in der aktuellen Architektur
# - Szenario-Unterordner: nur den aktiven anlegen (SZ_AKTIV aus config)
for d in [DIR_RAW, DIR_PROCESSED, DIR_INTERMEDIATE,
          os.path.join(DATA_FOLDER,'intermediate', SZ_AKTIV),   # nur aktiver Szenario-Ordner
          os.path.join(OUTPUT_FOLDER,'charts', SZ_AKTIV)]:       # Charts-Zielordner
    os.makedirs(d, exist_ok=True)

print(f'raw          : {os.path.abspath(DIR_RAW)}')
print(f'processed    : {os.path.abspath(DIR_PROCESSED)}')
print(f'intermediate : {os.path.abspath(DIR_INTERMEDIATE)}')

../sync/config.json geladen | MODE=data
Datenzeitraum: 2023–2026 | Jahre: [2023, 2024, 2025, 2026]
raw          : C:\Users\patri\Documents\docker\python\OWN\Grid-Arbitrage\data\raw
processed    : C:\Users\patri\Documents\docker\python\OWN\Grid-Arbitrage\data\processed
intermediate : C:\Users\patri\Documents\docker\python\OWN\Grid-Arbitrage\data\intermediate


**Hilfsfunktionen:** `log_dataindex()` schreibt jede erzeugte Datei ins Datenprotokoll;
`log_missing()` markiert fehlgeschlagene Downloads und schreibt `data/missing.txt`.


**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `log_dataindex` wird aus `lib/io_ops.py` importiert und
schreibt Einträge ins Daten-Provenienz-Protokoll `sync/dataindex.csv`.
Bereits bestehende Einträge für denselben Dateinamen werden als
`superseded` markiert. Aufklappbar darunter ist der Quellcode einsehbar.


In [5]:
show_source(log_dataindex)


<details>
<summary>🔎 Quellcode: <code>log_dataindex</code> (aus <code>lib/io_ops.py</code>)</summary>

```python
def log_dataindex(filename, source_url, local_path, data_type,
                  rows=None, size_kb=None, status='active', note='',
                  dataindex_path=None):
    """Schreibt einen Eintrag ins Daten-Provenienz-Protokoll.

    Existiert bereits ein aktiver Eintrag mit demselben ``filename``, wird
    dieser als ``superseded`` markiert (mit Zeitstempel in ``superseded_at``).

    Parameter
    ---------
    filename : str
        Dateiname (ohne Pfad).
    source_url : str
        Quelle (URL, Bibliotheksname, o.Ä.).
    local_path : str
        Relativer lokaler Pfad der Datei.
    data_type : {'raw','intermediate','processed','output'}
        Art der Datei in der Pipeline.
    rows : int, optional
        Anzahl Zeilen (für tabellarische Daten).
    size_kb : float, optional
        Grösse in Kilobyte (wird auf 1 Nachkommastelle gerundet).
    status : {'active','superseded','deleted'}, default 'active'
        Status des Eintrags.
    note : str, default ''
        Freitext-Kommentar.
    dataindex_path : str, optional
        Pfad zur ``dataindex.csv``. Wenn ``None``, wird im NB-Scope die
        globale Variable ``DATAINDEX`` gesucht (Rückwärtskompatibilität);
        Fallback ``"../sync/dataindex.csv"``.

    Return
    ------
    None. Schreibt nach ``dataindex.csv``.
    """
    import pandas as pd

    # dataindex_path auflösen
    if dataindex_path is None:
        # Versuche globale Variable DATAINDEX aus dem aufrufenden Scope
        import inspect
        caller_globals = inspect.stack()[1].frame.f_globals
        dataindex_path = caller_globals.get('DATAINDEX', '../sync/dataindex.csv')

    ts = datetime.utcnow().isoformat(timespec='seconds') + 'Z'

    if os.path.exists(dataindex_path):
        df_idx = pd.read_csv(dataindex_path)
        mask = (df_idx['filename'] == filename) & (df_idx['status'] == 'active')
        if mask.any():
            df_idx.loc[mask, 'status']         = 'superseded'
            df_idx.loc[mask, 'superseded_at']  = ts
    else:
        df_idx = pd.DataFrame(columns=DATAINDEX_COLUMNS)

    row = {
        'timestamp':      ts,
        'filename':       filename,
        'source_url':     source_url,
        'local_path':     local_path,
        'data_type':      data_type,
        'rows':           rows,
        'size_kb':        round(size_kb, 1) if size_kb else None,
        'status':         status,
        'superseded_at':  '',
        'note':           note,
    }
    pd.concat(
        [df_idx, pd.DataFrame([row])],
        ignore_index=True,
    ).to_csv(dataindex_path, index=False)

    print(f'  dataindex: {filename} [{status}]')
```

</details>


---
## Datensatz 1: ENTSO-E Day-Ahead Spot-Preise CH <a id='datensatz-1-entso-e-day-ahead-spot-preise-ch_NB_01'></a>

[↑ Inhaltsverzeichnis](#toc_NB_01)
    
**Quelle:** [ENTSO-E](../organisation/O_02_Glossar.ipynb#g-entsoe) Transparency Platform — `transparency.entsoe.eu`  
**Methode:** REST-[API](../organisation/O_02_Glossar.ipynb#g-api) via `entsoe-py` Python-Bibliothek (`query_day_ahead_prices`)  
**Zone:** `10YCH-SWISSGRIDZ` (Schweizer Bietungszone)  
**Format:** Stündliche Preise [EUR/MWh], [UTC](../organisation/O_02_Glossar.ipynb#g-utc)-Index  
**Zweck:** Hauptdatensatz für [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch)-Simulation und Wirtschaftlichkeitsrechnung.  
Der Intra-Tag-Spread (p75 − p25) ist der direkte Erlöstreiber der [Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage).


**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `fetch_entsoe_yearly` wird aus `lib/data_fetchers.py` importiert und
kapselt die jahresweise ENTSO-E-Abfrage mit 503-Retry. Die konkrete
Query (Preise, Last, Grenzflüsse, …) wird als Lambda übergeben —
so bleibt die Retry-Logik unverändert für neue Queries. Aufklappbar
ist der Quellcode einsehbar.


In [6]:
show_source(fetch_entsoe_yearly)


<details>
<summary>🔎 Quellcode: <code>fetch_entsoe_yearly</code> (aus <code>lib/data_fetchers.py</code>)</summary>

```python
def fetch_entsoe_yearly(query_fn, year, max_retries=3, wait_s=20,
                        tz='Europe/Zurich'):
    """Ruft eine ENTSO-E-Query jahresweise mit 503-Retry auf.

    ENTSO-E gibt bei Serverüberlastung HTTP 503 zurück. Jahresweiser Abruf
    mit Wartezeit zwischen Versuchen ist zuverlässiger als ein grosser
    Mehrjahresrequest.

    Parameter
    ---------
    query_fn : callable
        Funktion mit Signatur ``query_fn(start, end) -> result``. Typisch
        als Lambda: ``lambda s, e: client.query_day_ahead_prices('CH', start=s, end=e)``.
    year : int
        Jahr (z.B. 2023). Start = 1.1. 00:00, End = 31.12. 23:00 in ``tz``.
    max_retries : int, default 3
        Maximale Anzahl Versuche pro Jahr.
    wait_s : int, default 20
        Sekunden Pause zwischen Retries.
    tz : str, default 'Europe/Zurich'
        Timezone für die Jahres-Grenzen.

    Return
    ------
    Das Ergebnis von ``query_fn`` — typisch ein pandas DataFrame oder Series
    (abhängig von der ENTSO-E-Methode und -Version).

    Raises
    ------
    HTTPError
        Wenn nach ``max_retries`` weiterhin 503 kommt, oder bei anderen
        HTTP-Fehlern (z.B. 401 — ungültiger API-Key).

    Beispiele
    ---------
    Day-Ahead-Preise:

        client = EntsoePandasClient(api_key=key)
        ts = fetch_entsoe_yearly(
            lambda s, e: client.query_day_ahead_prices('CH', start=s, end=e),
            year=2023
        )

    Grenzflüsse CH -> DE:

        flows = fetch_entsoe_yearly(
            lambda s, e: client.query_crossborder_flows('CH', 'DE', start=s, end=e),
            year=2023
        )
    """
    import time
    import pandas as pd
    from requests.exceptions import HTTPError

    start = pd.Timestamp(f'{year}-01-01',       tz=tz)
    end   = pd.Timestamp(f'{year}-12-31 23:00', tz=tz)

    for attempt in range(1, max_retries + 1):
        try:
            return query_fn(start, end)
        except HTTPError as e:
            if '503' in str(e) and attempt < max_retries:
                print(f'  Jahr {year}: 503 → Versuch {attempt}/{max_retries}, '
                      f'warte {wait_s}s...')
                time.sleep(wait_s)
            else:
                raise  # Anderer Fehler oder max Retries erreicht
```

</details>


**🔎 Quellcode der importierten lib-Funktion**

`needs_download` aus `lib.io_ops` — aufklappbar ist der Quellcode einsehbar.


In [7]:
show_source(needs_download)


<details>
<summary>🔎 Quellcode: <code>needs_download</code> (aus <code>lib/io_ops.py</code>)</summary>

```python
def needs_download(path, min_kb, key, force_reload=None):
    """True wenn Datei fehlt, zu klein ist, oder force_reload[key] gesetzt.

    Typische Nutzung: vor externem Download entscheiden, ob Cache valide ist.

    Parameter
    ---------
    path : str
        Pfad zur Cache-Datei.
    min_kb : float
        Minimale erwartete Dateigrösse in KB.
    key : str
        Key für den force_reload-Dict (z.B. 'prices', 'netzlast').
    force_reload : dict, optional
        Dict mit ``{key: bool}``. Wenn ``None``, wird im Caller-Scope die
        globale Variable ``FORCE_RELOAD`` gesucht (aus config.json).

    Return
    ------
    bool
        True → Download nötig; False → Cache-Datei ist gut genug.
    """
    if force_reload is None:
        import inspect
        caller_globals = inspect.stack()[1].frame.f_globals
        force_reload = caller_globals.get('FORCE_RELOAD', {})

    if force_reload.get(key, False):
        print(f'  FORCE_RELOAD={key} → neu laden')
        return True
    if not os.path.exists(path):
        return True
    if os.path.getsize(path) < min_kb * 1024:
        return True
    return False
```

</details>


In [8]:
# API-Key aus ../sync/config.json (api_keys.entsoe) — nie im Code hardcoden
ENTSOE_API_KEY = (CFG.get('api_keys', {}).get('entsoe', '')
                  or os.environ.get('ENTSOE_API_KEY', ''))

PRICES_FILE = os.path.join(DIR_RAW, 'ch_spot_prices_raw.csv')

# ── Connectivity-Check (web-api Endpunkt, nicht nur Portal-Startseite) ───────
# Hinweis: transparency.entsoe.eu (HTTP 200) ≠ web-api.tp.entsoe.eu (kann 503 sein)
# Daher direkt den API-Endpunkt pingen statt die Webseite.
try:
    r = requests.get(
        'https://web-api.tp.entsoe.eu/api',
        params={'securityToken': ENTSOE_API_KEY, 'documentType': 'A44'},
        timeout=10)
    # 400 = Endpunkt erreichbar (Pflichtparameter fehlen — erwartetes Verhalten)
    # 401 = Endpunkt erreichbar, Key ungültig
    # 503 = Server temporär überlastet — ERREICHBAR, Retry übernimmt
    # → portal_ok = True für alle Codes ausser Netzwerkfehler
    portal_ok = r.status_code != 0
    print(f'ENTSO-E API-Endpunkt: HTTP {r.status_code} | '
          f'Key vorhanden: {bool(ENTSOE_API_KEY)}')
    if r.status_code == 503:
        print('  → 503: Server überlastet, aber erreichbar — Retry-Logik übernimmt.')
    elif r.status_code == 401:
        print('  → 401: API-Key ungültig — bitte prüfen.')
        portal_ok = False
except Exception as e:
    portal_ok = False
    print(f'ENTSO-E nicht erreichbar (Netzwerk?): {e}')


if not needs_download(PRICES_FILE, 10, 'spot_prices'):
    print(f'Preisdaten vorhanden ({os.path.getsize(PRICES_FILE)/1024:.0f} KB) – kein Re-Download.')
    df_prices = pd.read_csv(PRICES_FILE)

elif ENTSOE_API_KEY and portal_ok:
    print(f'Lade ENTSO-E CH Preise {START_YEAR}–{END_YEAR} ({len(YEARS)} Jahre, max. 3 Retries bei 503)...')
    from entsoe import EntsoePandasClient
    client = EntsoePandasClient(api_key=ENTSOE_API_KEY)
    frames = []
    for year in YEARS:
        print(f'  Lade {year}...', end=' ')
        ts_year = fetch_entsoe_yearly(
                lambda s, e: client.query_day_ahead_prices('CH', start=s, end=e),
                year)
        frames.append(ts_year)
        print(f'{len(ts_year):,} h OK')
    ts = pd.concat(frames).sort_index()
    ts = ts[~ts.index.duplicated(keep='first')]  # Überlapp-Schutz
    df_prices = ts.rename('price_eur_mwh').to_frame().reset_index()
    df_prices.columns = ['timestamp', 'price_eur_mwh']
    df_prices['timestamp'] = pd.to_datetime(df_prices['timestamp'], utc=True)
    df_prices.to_csv(PRICES_FILE, index=False, encoding='utf-8')
    kb = os.path.getsize(PRICES_FILE) / 1024
    log_dataindex('ch_spot_prices_raw.csv',
                  'https://transparency.entsoe.eu', PRICES_FILE, 'raw',
                  rows=len(df_prices), size_kb=kb, note=f'ENTSO-E API, entsoe-py, {START_YEAR}-{END_YEAR}')
    print(f'Gespeichert: {PRICES_FILE} | {len(df_prices):,} h | {kb:.0f} KB')
else:
    reason = 'Kein API-Key' if not ENTSOE_API_KEY else 'Portal/API nicht erreichbar (503?)'
    log_missing('ch_spot_prices_raw.csv', reason)
    raise RuntimeError(
        f'{reason}\n'
        '→ Bei 503: In 15-30 Min erneut versuchen (ENTSO-E hat gelegentlich Wartungsfenster).\n'
        '→ API-Key beantragen: transparency@entsoe.eu, Betreff: Restful API access\n'
        '→ Oder sofort weiterarbeiten: NB1b_Daten_Sim ausführen.')

print(f'Qualitaet: {len(df_prices):,} Zeilen | '
      f'Min: {df_prices["price_eur_mwh"].min():.1f} / '
      f'Max: {df_prices["price_eur_mwh"].max():.1f} EUR/MWh')


ENTSO-E API-Endpunkt: HTTP 400 | Key vorhanden: True
Lade ENTSO-E CH Preise 2023–2026 (4 Jahre, max. 3 Retries bei 503)...
  Lade 2023... 

8,759 h OK
  Lade 2024... 

8,783 h OK
  Lade 2025... 

8,759 h OK
  Lade 2026... 

2,783 h OK
  dataindex: ch_spot_prices_raw.csv [active]
Gespeichert: ../data\raw\ch_spot_prices_raw.csv | 29,084 h | 947 KB
Qualitaet: 29,084 Zeilen | Min: -427.5 / Max: 318.0 EUR/MWh


**Verifikation DS1:** Shape, Zeitraum und Wertebereich der Spot-Preise prüfen.


In [9]:
# ── Verifikation DS1: Spot-Preise ───────────────────────────────────────────
print(f'Shape   : {df_prices.shape}')
print(f'Zeitraum: {df_prices["timestamp"].min()} – {df_prices["timestamp"].max()}')
print(f'Nulls   : {df_prices.isnull().sum().sum()}')
print(f'Range   : {df_prices["price_eur_mwh"].min():.1f} – '
      f'{df_prices["price_eur_mwh"].max():.1f} EUR/MWh')
print(f"Pandas Version: {pd.__version__}")
df_prices.head(3)

Shape   : (29084, 2)
Zeitraum: 2022-12-31 23:00:00+00:00 – 2026-04-26 21:00:00+00:00
Nulls   : 0
Range   : -427.5 – 318.0 EUR/MWh
Pandas Version: 2.3.3


,timestamp,price_eur_mwh
0,2022-12-31 23:00:00+00:00,0.03
1,2023-01-01 00:00:00+00:00,-7.25
2,2023-01-01 01:00:00+00:00,-3.99


---
## Datensatz 2: CH Netzlast (Systemlast) <a id='datensatz-2-ch-netzlast-systemlast_NB_01'></a>

[↑ Inhaltsverzeichnis](#toc_NB_01)
    
**Quelle:** [ENTSO-E](../organisation/O_02_Glossar.ipynb#g-entsoe) Transparency Platform — `transparency.entsoe.eu`  
**Methode:** REST-[API](../organisation/O_02_Glossar.ipynb#g-api) via `entsoe-py` (`query_load('CH')`)  
**Identisch mit DS1:** Gleicher API-Key und Endpunkt — eine Moodle-URL für DS1+DS2.  
**Format:** Stündliche Systemlast [MW→GW], [UTC](../organisation/O_02_Glossar.ipynb#g-utc)-Index  
**Zweck:** Netzentlastungsmodell — aggregierte Batterien sollen die Spitzenlast senken.  
Koinzidenz mit Preishochpunkten bestätigt Business Case (Import teuer = Netz voll).


In [10]:
# ── Datensatz 2: CH Netzlast via ENTSO-E ────────────────────────────────────
# Gleicher API-Key wie für Preise.
# query_load('CH') liefert die stündliche Systemlast des Schweizer Regelblocks [MW].
# Jahresweiser Download mit Retry bei 503 — identisch zur DS1-Logik.

LOAD_FILE = os.path.join(DIR_RAW, 'ch_netzlast_raw.csv')


if not needs_download(LOAD_FILE, 10, 'netzlast'):
    print(f'Lastdaten vorhanden ({os.path.getsize(LOAD_FILE)/1024:.0f} KB) – kein Re-Download.')
    df_load = pd.read_csv(LOAD_FILE)

elif ENTSOE_API_KEY and portal_ok:
    print(f'Lade CH Netzlast {START_YEAR}–{END_YEAR} ({len(YEARS)} Jahre, jahresweise, max. 3 Retries bei 503)...')
    try:
        from entsoe import EntsoePandasClient
        client = EntsoePandasClient(api_key=ENTSOE_API_KEY)
        frames = []
        for year in YEARS:
            print(f'  Lade {year}...', end=' ')
            ts_year = fetch_entsoe_yearly(
                lambda s, e: client.query_load('CH', start=s, end=e),
                year)

            # query_load gibt je nach entsoe-py Version DataFrame oder Series zurück
            if isinstance(ts_year, pd.DataFrame):
                load_col = ts_year.columns[0]
                df_year  = ts_year[[load_col]].reset_index()
                df_year.columns = ['timestamp', 'load_mw']
            else:
                df_year = ts_year.to_frame(name='load_mw').reset_index()
                df_year.columns = ['timestamp', 'load_mw']

            frames.append(df_year)
            print(f'{len(df_year):,} h OK')

        df_load = pd.concat(frames, ignore_index=True)
        df_load = df_load[~df_load['timestamp'].duplicated(keep='first')]  # Überlapp-Schutz
        df_load['timestamp'] = pd.to_datetime(df_load['timestamp'], utc=True)
        df_load['load_gw']   = (df_load['load_mw'] / 1000).round(4)
        df_load = df_load[['timestamp', 'load_gw']].sort_values('timestamp').reset_index(drop=True)

        df_load.to_csv(LOAD_FILE, index=False, encoding='utf-8')
        kb = os.path.getsize(LOAD_FILE) / 1024
        log_dataindex(
            'ch_netzlast_raw.csv',
            'https://transparency.entsoe.eu (query_load CH)',
            LOAD_FILE, 'raw',
            rows=len(df_load), size_kb=kb,
            note=f'ENTSO-E query_load, jahresweise, MW→GW, {START_YEAR}–{END_YEAR}')
        print(f'Gespeichert: {LOAD_FILE} | {len(df_load):,} h | {kb:.0f} KB')

    except Exception as e:
        log_missing('ch_netzlast_raw.csv', str(e))
        raise RuntimeError(
            f'ENTSO-E query_load fehlgeschlagen: {e}\n'
            '→ NB01b_Daten_Sim ausführen (MODE=\'sim\' in ../sync/config.json setzen).')
else:
    reason = 'Kein API-Key' if not ENTSOE_API_KEY else 'Portal nicht erreichbar'
    log_missing('ch_netzlast_raw.csv', reason)
    raise RuntimeError(f'{reason} → NB01b_Daten_Sim ausführen.')

# Qualitätsprüfung: CH Systemlast typisch 5–12 GW
df_load_check = pd.read_csv(LOAD_FILE)
assert len(df_load_check) > 8000, f'Zu wenig Zeilen: {len(df_load_check)}'
assert df_load_check['load_gw'].between(2, 20).mean() > 0.98, 'Lastdaten ausserhalb plausiblem Bereich'
print(f'Qualität OK | Median: {df_load_check["load_gw"].median():.2f} GW | '
      f'{len(df_load_check):,} Zeilen')


Lade CH Netzlast 2023–2026 (4 Jahre, jahresweise, max. 3 Retries bei 503)...


  Lade 2023... 

8,759 h OK
  Lade 2024... 

8,783 h OK
  Lade 2025... 

8,759 h OK
  Lade 2026... 

2,769 h OK
  dataindex: ch_netzlast_raw.csv [active]
Gespeichert: ../data\raw\ch_netzlast_raw.csv | 29,070 h | 962 KB
Qualität OK | Median: 7.06 GW | 29,070 Zeilen


**Verifikation DS2:** Shape, Zeitraum und Wertebereich der Netzlast prüfen.


In [11]:
# ── Verifikation DS2: Netzlast ──────────────────────────────────────────────
print(f'Shape   : {df_load.shape}')
print(f'Zeitraum: {df_load["timestamp"].min()} – {df_load["timestamp"].max()}')
print(f'Nulls   : {df_load.isnull().sum().sum()}')
print(f'Range   : {df_load["load_gw"].min():.2f} – {df_load["load_gw"].max():.2f} GW')
df_load.head(3)


Shape   : (29070, 2)
Zeitraum: 2022-12-31 23:00:00+00:00 – 2026-04-26 09:00:00+00:00
Nulls   : 0
Range   : 2.74 – 10.44 GW


,timestamp,load_gw
0,2022-12-31 23:00:00+00:00,7.6668
1,2023-01-01 00:00:00+00:00,7.3730
2,2023-01-01 01:00:00+00:00,7.4643


---
## Fazit <a id='fazit_NB_01'></a>

[↑ Inhaltsverzeichnis](#toc_NB_01)

Mehrjährige stündliche CH-Daten geladen (Spot-Preise und Netzlast),
Abdeckung gemäss ENTSO-E-Verfügbarkeit. Saubere Ausgangsbasis für die Bereinigung
in NB02 und die Wirtschaftlichkeitsrechnung in NB03.

Der Datenzeitraum (`start_date`, `end_date`, `n_years`) ist in
`../sync/transfer.json` hinterlegt und wird von allen Downstream-Notebooks
per Transfer-Lader eingelesen.


---
## Abschluss <a id='abschluss_NB_01'></a>

[↑ Inhaltsverzeichnis](#toc_NB_01)

Datenzeitraum in `../sync/transfer.json` schreiben (SSOT für NB02–K_99),
dann Ausgabedateien auf Existenz und Mindestgrösse prüfen.


**Transfer NB01 → downstream:** Schreibt `datenzeitraum` (Start/End/n_years)
in `../sync/transfer.json` — Single Source of Truth für alle nachfolgenden Notebooks.


**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `save_transfer` wird aus `lib/io_ops.py` importiert und
in der folgenden Zelle verwendet. Aufklappbar ist der Quellcode einsehbar.


In [12]:
show_source(save_transfer)


<details>
<summary>🔎 Quellcode: <code>save_transfer</code> (aus <code>lib/io_ops.py</code>)</summary>

```python
def save_transfer(data, path='../sync/transfer.json', key=None):
    """Schreibt Daten nach transfer.json — mit Merge-Logik.

    Verhalten
    ---------
    * Wenn ``path`` existiert und nicht leer ist, wird das bestehende Dict
      geladen und mit den neuen Daten gemerged (bestehende andere Keys
      bleiben erhalten — wichtig für die Pipeline!).
    * Bei ``key=None`` muss ``data`` ein Dict sein und wird in die oberste
      Ebene gemerged (``existing.update(data)``).
    * Bei gegebenem ``key`` wird ``data`` unter diesem Top-Level-Key
      abgelegt (``existing[key] = data``).

    Parameter
    ---------
    data : dict oder any
        Zu schreibende Daten.
    path : str, default '../sync/transfer.json'
        Zieldatei.
    key : str, optional
        Top-Level-Key. Bei None muss ``data`` ein Dict sein.

    Return
    ------
    Das komplette Dict nach dem Write (nützlich für Chaining / Verifikation).
    """
    import json as _json

    # Bestehendes laden (wenn vorhanden)
    existing = {}
    if os.path.exists(path) and os.path.getsize(path) > 0:
        with open(path, encoding='utf-8') as _f:
            existing = _json.load(_f)

    # Mergen
    if key is None:
        if not isinstance(data, dict):
            raise TypeError(
                f"Bei key=None muss data ein dict sein, bekam {type(data).__name__}"
            )
        existing.update(data)
    else:
        existing[key] = data

    # Schreiben
    with open(path, 'w', encoding='utf-8') as _f:
        _json.dump(existing, _f, indent=2, ensure_ascii=False)

    return existing
```

</details>


In [13]:
# -- Transfer: Datenzeitraum in ../sync/transfer.json schreiben -----------------------
# Wird von NB02–K_99 gelesen; muss nach dem Laden der Preisdaten stehen,
# damit n_years aus echten Daten stammt.
_datenzeitraum = {
    'start_date': str(START_YEAR),
    'end_date':   str(END_YEAR),
    'n_years':    round(len(df_prices) / 8760, 3),
}
save_transfer(_datenzeitraum, key='datenzeitraum')
print(f"../sync/transfer.json: datenzeitraum {START_YEAR}–{END_YEAR} | "
      f"n_years={_datenzeitraum['n_years']:.3f}")


../sync/transfer.json: datenzeitraum 2023–2026 | n_years=3.320


**Abschlusskontrolle:** Pflicht-Ausgabedateien auf Existenz und Mindestgrösse prüfen;
`../sync/dataindex.csv`-Status anzeigen. Fehler müssen vor NB02 behoben werden.

**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `final_check` wird aus `lib/io_ops.py` importiert und
in der folgenden Zelle verwendet. Aufklappbar ist der Quellcode einsehbar.

In [14]:
show_source(final_check)

<details>
<summary>🔎 Quellcode: <code>final_check</code> (aus <code>lib/io_ops.py</code>)</summary>

```python
def final_check(nb_label, files=None, *, weiter_msg=None, fehler_msg=None,
                extras=None, show_dataindex=False,
                dataindex_path='../sync/dataindex.csv', width=60):
    """Standardisierte End-of-Notebook-Kontrolle für Pflicht- und Kür-NBs.

    Prüft Existenz und Mindestgrösse der angegebenen Output-Dateien,
    gibt formatiertes Resultat aus und liefert ``all_ok`` als Bool zurück.

    Parameter
    ---------
    nb_label : str
        Label des Notebooks im Output-Header, z.B. ``"NB01"``, ``"K_03"``.
    files : list of tuple, optional
        Zu prüfende Dateien als ``(path, label, min_bytes)``-Tuples.

        * ``min_bytes = 0`` → nur Existenz prüfen, Grösse nicht ausgeben
          (z.B. für PNG-Charts).
        * ``min_bytes > 0`` → zusätzlich Grösse prüfen und in KB/MB ausgeben
          (z.B. für CSV-Dateien).

        Bei ``files=None`` oder ``files=[]`` wird kein Check ausgeführt;
        die Funktion dient dann als reiner Status-Print (für Report-NBs
        ohne eigene Outputs wie K_00).
    weiter_msg : str, optional
        Nachricht für den Erfolgsfall, z.B. ``"NB02 Daten Bereinigung"``.
        Default: ``"nächstes Notebook"``.
    fehler_msg : str, optional
        Nachricht für den Fehlerfall (Kurzform, ohne "Fehler beheben vor").
        Default: identisch mit ``weiter_msg``.
    extras : list of str, optional
        Zusätzliche Print-Zeilen zwischen Datei-Check und Weiter-/Fehler-Hinweis.
        Sinnvoll für Kür-Hinweise oder Kontext.
    show_dataindex : bool, default False
        Wenn True, wird der aktive Auszug aus ``../sync/dataindex.csv`` ausgegeben.
        Typisch für NB01.
    dataindex_path : str, default '../sync/dataindex.csv'
        Pfad zur dataindex.csv (für ``show_dataindex=True``).
    width : int, default 60
        Breite der Trennlinie aus ``=``-Zeichen.

    Return
    ------
    bool
        ``True`` wenn alle Files existieren und Mindestgrösse erfüllen,
        ``False`` sonst. Bei ``files=None``/leer immer ``True``.
    """
    print(f'{nb_label} – Abschlusskontrolle')
    print('=' * width)

    all_ok = True

    if files:
        for path, label, min_bytes in files:
            exists = os.path.exists(path)
            size = os.path.getsize(path) if exists else 0
            ok = exists and size >= min_bytes

            if min_bytes > 0:
                size_str = _format_size(size) if exists else '   FEHLT'
                print(f'  {"✅" if ok else "❌"}  {label:<45} {size_str}')
            else:
                print(f'  {"✅" if ok else "❌"}  {label}')

            if not ok:
                all_ok = False

    if extras:
        if files:
            print()
        for line in extras:
            print(line)

    if show_dataindex and os.path.exists(dataindex_path):
        import pandas as pd
        df_idx = pd.read_csv(dataindex_path)
        active = df_idx[df_idx['status'] == 'active']
        print(f'\ndataindex.csv: {len(df_idx)} Einträge total, {len(active)} active')
        print(active[['filename', 'data_type', 'rows', 'size_kb', 'timestamp']]
              .to_string(index=False))

    print()
    weiter = weiter_msg or 'nächstes Notebook'
    fehler = fehler_msg or weiter
    if all_ok:
        print(f'→ Weiter mit {weiter}.')
    else:
        print(f'→ Fehler beheben vor {fehler}.')

    return all_ok
```

</details>


In [15]:
# ── Abschlusskontrolle NB01 ──────────────────────────────────────────────────
final_check(
    'NB01',
    files=[
        (os.path.join(DIR_RAW, 'ch_spot_prices_raw.csv'),
         'ch_spot_prices_raw.csv (DS1) — ENTSO-E Spot-Preise', 10_240),
        (os.path.join(DIR_RAW, 'ch_netzlast_raw.csv'),
         'ch_netzlast_raw.csv (DS2) — ENTSO-E Netzlast',      10_240),
    ],
    weiter_msg='NB02 Daten Bereinigung',
    fehler_msg='NB02',
    extras=['Kür-Datensätze: werden in K_01 / K_02 bei Bedarf geladen.'],
    show_dataindex=True,
)

NB01 – Abschlusskontrolle


  ✅  ch_spot_prices_raw.csv (DS1) — ENTSO-E Spot-Preise   946.5 KB
  ✅  ch_netzlast_raw.csv (DS2) — ENTSO-E Netzlast    961.9 KB

Kür-Datensätze: werden in K_01 / K_02 bei Bedarf geladen.

dataindex.csv: 2 Einträge total, 2 active
              filename data_type  rows  size_kb            timestamp
ch_spot_prices_raw.csv       raw 29084    946.5 2026-04-26T10:51:29Z
   ch_netzlast_raw.csv       raw 29070    961.9 2026-04-26T10:51:45Z

→ Weiter mit NB02 Daten Bereinigung.


True

|  | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [NB02 Daten Bereinigung →](02_Daten_Bereinigung.ipynb) |
|:---|:---:|---:|
